# Poker McPokerface — Bot Arena

Automated bot-vs-bot tournaments. The runner plays N hands, wraps each bot in a `TrackingAgent` so all reasoning is captured to disk, and writes append-only JSONL logs into `runs/<session_id>/` for the analytics notebook to consume.

Right now the field is filled with `MockBot`s — the engine knows nothing about that, so once the LLM-backed bots in `bots/ollama_bot.py` are wired up, you swap the `MockBot(...)` constructors for `OllamaBot(...)` and rerun.

## 1. Imports + path setup

In [ ]:
import sys
from pathlib import Path
# Notebooks live in `notebooks/`. Make sure the project root is on
# sys.path so we can `import engine`, `import bots`, etc.
_here = Path.cwd()
_root = _here.parent if _here.name == 'notebooks' else _here
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
print(f'Project root: {_root}')

In [ ]:
from bots import MockBot, Personality
from runner import RunnerConfig, TournamentRunner

## 2. Define personalities + a roster of bots

Four bots, two personalities × two 'models'. This is the minimum that lets the analytics notebook produce non-trivial groupbys for each of the brief's three questions.

In [ ]:
aggressive = Personality(
    id='aggressive',
    description='Raises whenever possible.',
    system_prompt='You are an aggressive 5-card draw poker player.',
)
cautious = Personality(
    id='cautious',
    description='Calls and checks rather than raising.',
    system_prompt='You are a cautious 5-card draw poker player.',
)

raise_resp = '{"reasoning": "I always raise.", "action": "raise", "amount": 5}'
call_resp  = '{"reasoning": "I always call.", "action": "call"}'
stand_pat  = '{"reasoning": "Standing pat.", "discards": []}'

bots = [
    MockBot('Llama-Aggro', aggressive, raise_resp, stand_pat, model_id='mock-llama'),
    MockBot('Llama-Calm',  cautious,   call_resp,  stand_pat, model_id='mock-llama'),
    MockBot('Mistral-Aggro', aggressive, raise_resp, stand_pat, model_id='mock-mistral'),
    MockBot('Mistral-Calm',  cautious,   call_resp,  stand_pat, model_id='mock-mistral'),
]

for b in bots:
    print(f"  {b.name:<16s}  model={b.model_id:<14s} personality={b.personality.id}")

## 3. Configure + run the tournament

`broke_player_policy='rebuy'` keeps every bot in the data set every hand by topping their stack back to `starting_chips` between hands when they would otherwise miss the ante. Use `'elimination'` for a true knockout-style run.

In [ ]:
cfg = RunnerConfig(
    num_hands=50,
    starting_chips=200,
    ante=2,
    min_bet=5,
    broke_player_policy='rebuy',
    seed=42,
    verbose=False,           # flip to True for per-hand chatter
    # Absolute path → all sessions land in <project_root>/runs/
    # regardless of where Jupyter was launched from. The
    # analytics notebook looks in the same place.
    sessions_root=str(_root / 'runs'),
)

result = TournamentRunner(bots, cfg).run()
print(result.summary_table())
print()
print(f'Logs at: {result.session_dir}')

## 4. Peek at what was logged

A spot-check that the JSONL files exist and contain what we expect. The analytics notebook will pick these up and run the brief's three required aggregations on them.

In [ ]:
from tracker import load_hands, load_actions, load_reasoning, load_config

config    = load_config(result.session_dir)
hands     = load_hands(result.session_dir)
actions   = load_actions(result.session_dir)
reasoning = load_reasoning(result.session_dir)

print(f'config.json   started_at={config["started_at"]}')
print(f'              ended_at  ={config["ended_at"]}')
print(f'              seats     ={[s["name"] for s in config["seats"]]}')
print(f'hands.jsonl     {len(hands):>5d} rows')
print(f'actions.jsonl   {len(actions):>5d} rows')
print(f'reasoning.jsonl {len(reasoning):>5d} rows')

## 5. Next step

Open **`analytics.ipynb`** in this folder to load the session you just produced and run the brief's three groupbys plus chip-trend charts.